Part 1: Environment Setup and Path Configuration

In [ ]:
import pandas as pd
import numpy as np
import ast
from pathlib import Path
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier
import sklearn_crfsuite
from nltk.tokenize.treebank import TreebankWordDetokenizer

BASE_PATH = Path(r'C:\Users\Lenovo\Desktop\NLP project\ebm_nlp_2_00\axis2_B')
INPUT_CSV = BASE_PATH / "outputs" / "sentence_records_df.csv"
OUTPUT_DIR = BASE_PATH / "outputs"

def safe_eval(val):
    if isinstance(val, list): return val
    try: return ast.literal_eval(val)
    except: return []

df = pd.read_csv(INPUT_CSV)
df['sentence_tokens'] = df['sentence_tokens'].apply(safe_eval)

all_docs = df['doc_id'].unique()
test_docs = all_docs[:191] 

test_df = df[df['doc_id'].isin(test_docs)].copy()
train_df = df[~df['doc_id'].isin(test_docs)].copy()

print(f"Total Sentences: {len(df)}")
print(f"Training on: {len(train_df)} sentences")
print(f"Testing on: {len(test_df)} sentences")

Total Sentences: 50107
Training on: 47895 sentences
Testing on: 2212 sentences


Step 2: Sentence Embedding (PubMedBERT)

In [7]:
embed_model = SentenceTransformer('microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext')

X_train_embeddings = embed_model.encode(train_df['sentence_text'].tolist(), show_progress_bar=True)
X_test_embeddings = embed_model.encode(test_df['sentence_text'].tolist(), show_progress_bar=True)

No sentence-transformers model found with name microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext. Creating a new one with mean pooling.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1497 [00:00<?, ?it/s]

Batches:   0%|          | 0/70 [00:00<?, ?it/s]

Step 3: CRF Feature Engineering

In [8]:
def word2features(sent_tokens, i):
    word = str(sent_tokens[i])
    features = {
        'bias': 1.0,
        'word.lower()': word.lower(),
        'word[-3:]': word[-3:],
        'word.isupper()': word.isupper(),
        'word.isdigit()': word.isdigit(),
    }
    if i > 0:
        features['-1:word.lower()'] = str(sent_tokens[i-1]).lower()
        if i > 1:
            features['-2:word.lower()'] = str(sent_tokens[i-2]).lower()
    else:
        features['BOS'] = True
    if i < len(sent_tokens) - 1:
        features['+1:word.lower()'] = str(sent_tokens[i+1]).lower()
        if i < len(sent_tokens) - 2:
            features['+2:word.lower()'] = str(sent_tokens[i+2]).lower()
    else:
        features['EOS'] = True
    return features

Step 4: Hybrid Model Training and Prediction

In [29]:
import pandas as pd
import re
import nltk
from nltk.tokenize.treebank import TreebankWordDetokenizer
from sklearn.ensemble import RandomForestClassifier
import sklearn_crfsuite

nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

def get_pico_score_v2(phrase):
    p = phrase.lower()
    scores = {'P': 0, 'I': 0, 'O': 0}
    
    if any(k in p for k in ['patient', 'subject', 'child', 'adult', 'volunteer', 'smoker', 'asthmatic', 'women', 'men']):
        scores['P'] += 5
    if any(p.endswith(s) for s in ['itis', 'osis', 'emia', 'pathy', 'disease', 'syndrome']):
        scores['P'] += 4
    
    if any(k in p for k in ['mg', 'ml', 'dose', 'therapy', 'treatment', 'drug', 'surgery', 'tablet', 'injection', 'infusion', 'placebo', 'administration']):
        scores['I'] += 5
    if any(re.search(rf"{s}\b", p) for s in ['azole', 'idine', 'mycin', 'cillin', 'olol', 'pril', 'sartan', 'mab']):
        scores['I'] += 7
    
    if any(k in p for k in ['rate', 'ratio', 'score', 'scale', 'level', 'mortality', 'survival', 'incidence', 'reduction', 'change', 'increase', 'decrease']):
        scores['O'] += 6
    if re.search(r'p\s*[<>=]|mean|sd|95%|ci\b', p):
        scores['O'] += 8

    if any(k in p for k in ['mg', 'dose', 'administration', 'tablet']):
        scores['P'] -= 10
    if any(k in p for k in ['rate', 'p =', 'p <']):
        scores['P'] -= 8
    
    return scores

def final_clean_pico(phrases, target_type, max_words=6):
    cleaned = []
    academic_noise = {'evaluate', 'compare', 'study', 'report', 'randomize', 'assigned', 'background', 'results', 'methods', 'conclusions'}
    
    for p in phrases:
        p_low = p.lower().strip()
        if len(p_low) < 3 or any(n in p_low for n in academic_noise):
            continue
        if len(p_low.split()) > max_words:
            continue
        
        scores = get_pico_score_v2(p_low)
        if scores[target_type] > 0 and scores[target_type] == max(scores.values()):
            cleaned.append(p)
            
    unique = sorted(list(set(cleaned)), key=len, reverse=True)
    final = []
    for p in unique:
        if not any(p.lower() in other.lower() and p.lower() != other.lower() for other in final):
            final.append(p)
    return "; ".join(final[:5])

final_output = pd.DataFrame({'doc_id': test_df['doc_id'].unique()})
detokenizer = TreebankWordDetokenizer()

pico_configs = [
    {'col': 'p_count', 'pred': 'participants_pred', 'type': 'P', 'limit': 0.35},
    {'col': 'i_count', 'pred': 'interventions_pred', 'type': 'I', 'limit': 0.45},
    {'col': 'o_count', 'pred': 'outcomes_pred', 'type': 'O', 'limit': 0.45}
]

for cfg in pico_configs:
    target, pred_col, p_type, limit = cfg['col'], cfg['pred'], cfg['type'], cfg['limit']
    print(f"Refining Pipeline for {p_type}...")
    
    y_train_rf = (train_df[target] > 0).astype(int)
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1).fit(X_train_embeddings, y_train_rf)
    
    train_subset = train_df[train_df[target] > 0]
    X_crf_train = [[word2features(r['sentence_tokens'], i) for i in range(len(r['sentence_tokens']))] for _, r in train_subset.iterrows()]
    y_crf_train = []
    for _, r in train_subset.iterrows():
        tags = nltk.pos_tag(r['sentence_tokens'])
        y_crf_train.append(['I' if (tag.startswith(('NN', 'JJ', 'CD')) and len(word) > 1) else 'O' for word, tag in tags])
    
    crf = sklearn_crfsuite.CRF(algorithm='lbfgs', max_iterations=100).fit(X_crf_train, y_crf_train)
    
    test_df['prob'] = rf.predict_proba(X_test_embeddings)[:, 1]
    
    doc_results = []
    for d_id, group in test_df.groupby('doc_id'):
        extracted = []
        top_sents = group.nlargest(3, 'prob')
        for _, row in top_sents[top_sents['prob'] >= limit].iterrows():
            feats = [word2features(row['sentence_tokens'], i) for i in range(len(row['sentence_tokens']))]
            tags = crf.predict_single(feats) 
            phrase = []
            for i, tag in enumerate(tags):
                if tag == 'I':
                    phrase.append(row['sentence_tokens'][i])
                else:
                    if phrase:
                        extracted.append(detokenizer.detokenize(phrase))
                        phrase = []
            if phrase:
                extracted.append(detokenizer.detokenize(phrase))
        
        doc_results.append({'doc_id': d_id, pred_col: final_clean_pico(extracted, p_type)})
    
    final_output = final_output.merge(pd.DataFrame(doc_results), on='doc_id', how='left')

final_output.fillna("").to_csv(OUTPUT_DIR / "Axis2B_RESULTS2.csv", index=False)
print("Pipeline Success. Result: Axis2B_RESULTS2.csv")

Refining Pipeline for P...
Refining Pipeline for I...
Refining Pipeline for O...
Pipeline Success. Result: Axis2B_RESULTS2.csv
